# NBA Player Value Analysis — Data Cleaning Pipeline
This notebook loads, cleans, and merges six seasons (2021–2026) of NBA player data
from Basketball Reference and HoopsHype into a single master dataset used for all
analysis in `analysis.ipynb`.

**Sources:**
- Basketball Reference — Advanced stats and per game stats (2021–2026)
- HoopsHype — Player salary data (2021–2025)
- Basketball Reference — Player salary data (2026)

**Output:** `cleaned/master_final.csv` — 2,068 rows x 64 columns

## 1. Load Raw Data
Load advanced stats and per game stats for all six seasons and tag each row with its season year.

In [1]:
import pandas as pd
import os

# Load Advanced Stats
advanced_frames = []
for year in range(2021, 2027):
    df = pd.read_csv(f"raw/advanced_{year}.csv")
    df["season"] = year
    advanced_frames.append(df)
advanced = pd.concat(advanced_frames, ignore_index=True)

# Load Per Game Stats
per_game_frames = []
for year in range(2021, 2027):
    df = pd.read_csv(f"raw/per_game_{year}.csv")
    df["season"] = year
    per_game_frames.append(df)
per_game = pd.concat(per_game_frames, ignore_index=True)

print(f"Advanced rows loaded: {advanced.shape[0]}")
print(f"Per game rows loaded: {per_game.shape[0]}")

Advanced rows loaded: 4405
Per game rows loaded: 4405


## 2. Handle Traded Players
Players who were traded mid-season appear multiple times — once per team and once as
a combined total row (2TM, 3TM, 4TM). We keep only the combined row to avoid
double-counting their season stats.

In [2]:
def clean_traded(df):
    traded_mask = df["Team"].isin(["2TM", "3TM", "4TM"])
    traded_rows = df[traded_mask].copy()
    non_traded_rows = df[~traded_mask].copy()
    traded_pairs = set(zip(traded_rows["Player"], traded_rows["season"]))
    non_traded_rows = non_traded_rows[
        ~non_traded_rows.apply(lambda row: (row["Player"], row["season"]) in traded_pairs, axis=1)
    ]
    return pd.concat([traded_rows, non_traded_rows], ignore_index=True)

advanced_clean = clean_traded(advanced)
per_game_clean = clean_traded(per_game)

print(f"Advanced — before: {advanced.shape[0]}, after: {advanced_clean.shape[0]}")
print(f"Per game — before: {per_game.shape[0]}, after: {per_game_clean.shape[0]}")

Advanced — before: 4405, after: 3413
Per game — before: 4405, after: 3413


## 3. Apply Minutes Filter
Restrict to players with at least 500 minutes played to exclude garbage time
and small sample observations from distorting the analysis.

In [3]:
advanced_clean["MP"] = pd.to_numeric(advanced_clean["MP"], errors="coerce")
per_game_clean["MP"] = pd.to_numeric(per_game_clean["MP"], errors="coerce")

advanced_clean = advanced_clean[advanced_clean["MP"] >= 500].reset_index(drop=True)

# Filter per game to match valid advanced player-season pairs
valid_pairs = set(zip(advanced_clean["Player"], advanced_clean["season"]))
per_game_clean = per_game_clean[
    per_game_clean.apply(lambda row: (row["Player"], row["season"]) in valid_pairs, axis=1)
].reset_index(drop=True)

print(f"Advanced after minutes filter: {advanced_clean.shape[0]} rows")
print(f"Per game after minutes filter: {per_game_clean.shape[0]} rows")

Advanced after minutes filter: 2218 rows
Per game after minutes filter: 2218 rows


## 4. Merge Advanced and Per Game Stats
Inner join on 'Player' and 'season' so only rows present in both datasets are kept.

In [4]:
master = pd.merge(
    advanced_clean,
    per_game_clean,
    on=["Player", "season"],
    how="inner",
    suffixes=("_adv", "_pg")
)

print(f"Master shape: {master.shape}")
print(f"Unique players: {master['Player'].nunique()}")
print(f"Seasons covered: {sorted(master['season'].unique())}")

Master shape: (2218, 62)
Unique players: 676
Seasons covered: [2021, 2022, 2023, 2024, 2025, 2026]


## 5. Load and Clean Salary Data
Salary data comes from two sources: HoopsHype (2021–2025) and Basketball Reference
(2026). Both are standardized to a common format before merging.

In [5]:
salary_frames = []

for year in range(2021, 2026):
    df = pd.read_csv(f"raw/salary_{year}.csv")
    df = df[["Player", "Salary"]].copy()
    df["season"] = year
    salary_frames.append(df)

sal_2026 = pd.read_csv("raw/salary_2026.csv", skiprows=2, header=0)
sal_2026 = sal_2026[["Unnamed: 0", "Unnamed: 1"]].copy()
sal_2026.columns = ["Player", "Salary"]
sal_2026["Salary"] = sal_2026["Salary"].replace(r'[\$,]', '', regex=True).str.strip()
sal_2026["season"] = 2026
salary_frames.append(sal_2026)

salary = pd.concat(salary_frames, ignore_index=True)
salary = salary.dropna(subset=["Player", "Salary"])
salary = salary[salary["Player"] != ""]
salary["Salary"] = pd.to_numeric(salary["Salary"], errors="coerce")
salary = salary.dropna(subset=["Salary"])

print(f"Salary rows loaded: {salary.shape[0]}")
print(f"Seasons: {sorted(salary['season'].unique())}")

Salary rows loaded: 3444
Seasons: [2021, 2022, 2023, 2024, 2025, 2026]


## 6. Normalize Player Names and Merge Salary
Player names differ slightly between sources due to accents, suffixes, and
punctuation. Names are normalized before merging to maximize match rate.

In [6]:
import unicodedata
import re

name_map = {
    "lou williams": "louis williams",
    "bill nwob": "bill nwoboshi",
    "mfiondu kabengele": "maurice harkless",
    "nah'shon hyland": "bones hyland",
    "devonte' graham": "devonte graham",
    "deandre' bembry": "deandre bembry",
    "juan toscano-anderson": "juan toscano anderson",
    "russell westbrook": "russell westbrook",
}

def normalize_name(name):
    if pd.isna(name):
        return name
    name = unicodedata.normalize("NFKD", str(name))
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = name.lower().strip()
    name = re.sub(r'\b(jr\.?|sr\.?|ii|iii|iv)\b', '', name)
    name = name.replace(".", "").replace("'", "")
    name = re.sub(r'\s+', ' ', name).strip()
    return name_map.get(name, name)

master["Player_norm"] = master["Player"].apply(normalize_name)
salary["Player_norm"] = salary["Player"].apply(normalize_name)

master_with_salary = pd.merge(
    master,
    salary[["Player_norm", "season", "Salary"]],
    on=["Player_norm", "season"],
    how="left"
)

matched = master_with_salary["Salary"].notna().sum()
unmatched = master_with_salary["Salary"].isna().sum()
print(f"Matched: {matched} | Unmatched: {unmatched} | Match rate: {matched/len(master_with_salary)*100:.1f}%")

Matched: 2068 | Unmatched: 226 | Match rate: 90.1%


## 7. Calculate Cap Percentage and Save
Drop unmatched rows, calculate each player's salary as a percentage of that
season's NBA salary cap, and save the final master dataset.

In [7]:
master_final = master_with_salary.dropna(subset=["Salary"]).reset_index(drop=True)

cap_by_season = {
    2021: 109140000,
    2022: 112414000,
    2023: 123655000,
    2024: 136021000,
    2025: 140588000,
    2026: 141000000
}

master_final["Cap_Pct"] = master_final.apply(
    lambda row: row["Salary"] / cap_by_season[row["season"]], axis=1
)

master_final = master_final.drop(columns=["Player_norm"])

os.makedirs("cleaned", exist_ok=True)
master_final.to_csv("cleaned/master_final.csv", index=False)

print(f"Final shape: {master_final.shape}")
print(f"Unique players: {master_final['Player'].nunique()}")
print(f"Saved to cleaned/master_final.csv")

Final shape: (2068, 64)
Unique players: 623
Saved to cleaned/master_final.csv
